In [1]:
!pip install -q requests scipy numpy

In [2]:
import requests
import numpy as np
import json
import time
from typing import List, Tuple, Dict
from scipy import stats
from collections import Counter, defaultdict
import random
from tqdm import tqdm

In [3]:
# Configuration
API_KEY = "wCew7zRgfX9wsulqo5P5yx-xWO2gIyu-lQgvif4c1xc7yJGTCix5Osc1NdxlRQKvo8U"
BASE_URL = "https://isl-llms.ethz.ch"
API_URL = BASE_URL + "/api"

# Rate limiting
DAILY_LIMIT = 3000
TOTAL_LIMIT = 12000
query_count = {"pvalue": 0, "bool": 0}
VERBOSE = True

In [64]:
def check_quota(api_key: str):
  url = "https://isl-llms.ethz.ch/check-limits/{}".format(API_KEY)
  headers = {"accept": "application/json"}
  try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    print(response.text)
    return response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error checking quota: {e}")
    return None
check_quota(API_KEY)

{"user":"ribadov@student.ethz.ch","bool_left_today":1723,"pvalue_left_today":3000,"bool_left_total":7711,"pvalue_left_total":4822,"bool_day_reset_at":"2026-06-06 15:17:48 CEST","pvalue_day_reset_at":"window not started — next request will start a fresh 24h window"}


{'user': 'ribadov@student.ethz.ch',
 'bool_left_today': 1723,
 'pvalue_left_today': 3000,
 'bool_left_total': 7711,
 'pvalue_left_total': 4822,
 'bool_day_reset_at': '2026-06-06 15:17:48 CEST',
 'pvalue_day_reset_at': 'window not started — next request will start a fresh 24h window'}

In [30]:
def get_tokens():
    url_get_tokens = API_URL + "/tokens"

    payload = {}
    headers = {
      'x-api-key': API_KEY
    }

    return requests.request("GET", url_get_tokens, headers=headers, data=payload).json()

In [31]:
tokens = get_tokens()

In [32]:
# Get the token ids for each endpoint. Remember you must use them with the corect endpoint!
pvalue_token_ids = [i for i in tokens if i["endpoint"]=="pvalue"]
bool_token_ids = [i for i in tokens if i["endpoint"]=="bool"]
# Get tokens for the pvalue endpoint
p_value_tokens = [d["tokens"] for d in pvalue_token_ids]
bool_tokens = [d["tokens"] for d in bool_token_ids]

In [33]:
pvalue_token_ids

[{'tokens': '2324,15,187,50273,94,50276,605,12953,492,81,1306,27,70,33,681,16,1206,4904,22434,757,14,19,69,20,72,18,5840,9194,89,26,2874,71,22,14836,38510,1796,21,67,23,8407,24,20450,1525,14576,12847,1235,3832,68,2945,1257,25,3342,1867,2320,17,264,1099,2504,1857,66,1762,746,1630,12964,3953,23735,4989,1812,1619,1717,1549,17984,453,3583,6357,1229,7332,365,1508,15054,24117,2222,2358,3121,1976,3046,1047,1010,34,2941,883,2691,39,4529,2446,35,13482,20926,1678,740,2090,13743,1348,4148,1839,3566,805,1540,1012,2227,1036,1237,1934,38,6327,14711,317,40629,5781,12522,3348,24235,1967,1438,3439,1731,567,2693,11395,357,3763',
  'endpoint': 'pvalue'},
 {'tokens': '15432,352,369,3986,1598,7660,26474,13,247,27204,326,6221,275,271,12296,13670,3379,494,3558,323,897,327,253,3345,273,581,2720,432,534,597,497,1160,281,616,1073,13083,830,28,949,824,2758,347,594,1142,18010,10604,10985,285,9046,17080,407,3473,21169,314,387,697,2553,15,380,3678,908,310,9619,4872,342,1016,9402,1146,3809,2217,8136,2112,521,9881,76

In [34]:
bool_token_ids

[{'tokens': '2324,15,1916,2319,841,3533,13,359,452,3559,2067,9938,342,253,6508,370,57,5,285,5469,3786,1008,39962,28,1214,1704,3305,1570,380,1551,273,436,2929,310,10932,347,3637,8048,187,20197,60,510,6880,281,247,28095,2919,313,66,10,10140,2367,2407,92,81,9496,11,724,387,669,3442,61,3151,16,19,393,620,374,1352,1181,535,50275,6407,14,13921,496,4149,50264,11592,18743,444,14398,1159,50254,979,50265,209,16731,78,82,13488,50274,3138,5695,16943,3429,30,17,89,740,67,64,68,18,33,85,324,11828,2582,8155,9,45,5414,35,37077,1447,859,696,905,3671,88,20,29013,50266,30183,835,642,16841,4620,327,390',
  'endpoint': 'bool'},
 {'tokens': '15432,327,760,581,990,273,253,2613,2553,275,247,973,342,271,12296,17568,5939,2190,1016,824,326,13,7293,2220,616,9544,475,72,2992,597,452,642,1246,11361,15,187,510,1894,2182,3000,346,11,88,1582,598,432,1529,3,403,908,347,6667,323,1650,672,12930,849,281,5513,285,1480,1055,272,436,2238,1171,1789,390,9364,16,40651,875,1110,2426,762,534,627,320,3559,313,30786,685,3365,1146,3

In [35]:
p_value_tokens

['2324,15,187,50273,94,50276,605,12953,492,81,1306,27,70,33,681,16,1206,4904,22434,757,14,19,69,20,72,18,5840,9194,89,26,2874,71,22,14836,38510,1796,21,67,23,8407,24,20450,1525,14576,12847,1235,3832,68,2945,1257,25,3342,1867,2320,17,264,1099,2504,1857,66,1762,746,1630,12964,3953,23735,4989,1812,1619,1717,1549,17984,453,3583,6357,1229,7332,365,1508,15054,24117,2222,2358,3121,1976,3046,1047,1010,34,2941,883,2691,39,4529,2446,35,13482,20926,1678,740,2090,13743,1348,4148,1839,3566,805,1540,1012,2227,1036,1237,1934,38,6327,14711,317,40629,5781,12522,3348,24235,1967,1438,3439,1731,567,2693,11395,357,3763',
 '15432,352,369,3986,1598,7660,26474,13,247,27204,326,6221,275,271,12296,13670,3379,494,3558,323,897,327,253,3345,273,581,2720,432,534,597,497,1160,281,616,1073,13083,830,28,949,824,2758,347,594,1142,18010,10604,10985,285,9046,17080,407,3473,21169,314,387,697,2553,15,380,3678,908,310,9619,4872,342,1016,9402,1146,3809,2217,8136,2112,521,9881,762,5368,13100,723,19693,2710,9968,3082,313,70,83

In [36]:
bool_tokens

['2324,15,1916,2319,841,3533,13,359,452,3559,2067,9938,342,253,6508,370,57,5,285,5469,3786,1008,39962,28,1214,1704,3305,1570,380,1551,273,436,2929,310,10932,347,3637,8048,187,20197,60,510,6880,281,247,28095,2919,313,66,10,10140,2367,2407,92,81,9496,11,724,387,669,3442,61,3151,16,19,393,620,374,1352,1181,535,50275,6407,14,13921,496,4149,50264,11592,18743,444,14398,1159,50254,979,50265,209,16731,78,82,13488,50274,3138,5695,16943,3429,30,17,89,740,67,64,68,18,33,85,324,11828,2582,8155,9,45,5414,35,37077,1447,859,696,905,3671,88,20,29013,50266,30183,835,642,16841,4620,327,390',
 '15432,327,760,581,990,273,253,2613,2553,275,247,973,342,271,12296,17568,5939,2190,1016,824,326,13,7293,2220,616,9544,475,72,2992,597,452,642,1246,11361,15,187,510,1894,2182,3000,346,11,88,1582,598,432,1529,3,403,908,347,6667,323,1650,672,12930,849,281,5513,285,1480,1055,272,436,2238,1171,1789,390,9364,16,40651,875,1110,2426,762,534,627,320,3559,313,30786,685,3365,1146,3542,481,380,6158,476,2299,2395,11794,604,686,

In [37]:
# You can obtain a list of integers by using the following function
def tokens_to_array(token_list_str: str) -> np.ndarray:
    return np.array([int(i) for i in token_list_str.split(",")])

tokens_to_array(bool_token_ids[0]["tokens"])

array([ 2324,    15,  1916,  2319,   841,  3533,    13,   359,   452,
        3559,  2067,  9938,   342,   253,  6508,   370,    57,     5,
         285,  5469,  3786,  1008, 39962,    28,  1214,  1704,  3305,
        1570,   380,  1551,   273,   436,  2929,   310, 10932,   347,
        3637,  8048,   187, 20197,    60,   510,  6880,   281,   247,
       28095,  2919,   313,    66,    10, 10140,  2367,  2407,    92,
          81,  9496,    11,   724,   387,   669,  3442,    61,  3151,
          16,    19,   393,   620,   374,  1352,  1181,   535, 50275,
        6407,    14, 13921,   496,  4149, 50264, 11592, 18743,   444,
       14398,  1159, 50254,   979, 50265,   209, 16731,    78,    82,
       13488, 50274,  3138,  5695, 16943,  3429,    30,    17,    89,
         740,    67,    64,    68,    18,    33,    85,   324, 11828,
        2582,  8155,     9,    45,  5414,    35, 37077,  1447,   859,
         696,   905,  3671,    88,    20, 29013, 50266, 30183,   835,
         642, 16841,

In [38]:
# Use this function to transform a list of integers into a valid string you can submit through the API
def tokens_to_string(token_list: np.array):
    return ",".join([str(i) for i in token_list])

tokens_to_string(tokens_to_array(bool_token_ids[0]["tokens"]))

'2324,15,1916,2319,841,3533,13,359,452,3559,2067,9938,342,253,6508,370,57,5,285,5469,3786,1008,39962,28,1214,1704,3305,1570,380,1551,273,436,2929,310,10932,347,3637,8048,187,20197,60,510,6880,281,247,28095,2919,313,66,10,10140,2367,2407,92,81,9496,11,724,387,669,3442,61,3151,16,19,393,620,374,1352,1181,535,50275,6407,14,13921,496,4149,50264,11592,18743,444,14398,1159,50254,979,50265,209,16731,78,82,13488,50274,3138,5695,16943,3429,30,17,89,740,67,64,68,18,33,85,324,11828,2582,8155,9,45,5414,35,37077,1447,859,696,905,3671,88,20,29013,50266,30183,835,642,16841,4620,327,390'

In [39]:
assert tokens_to_string(tokens_to_array(bool_token_ids[0]["tokens"])) == (bool_token_ids[0]["tokens"])

In [40]:
# Use this function to validate your tokens are valid
def validate_tokens(token_list_str: str):
    assert isinstance(token_list_str, str), "Your list must be a string of comma-separated integers"
    assert len(token_list_str)>0, "Your list is empty"

    # Split the string by commas
    parts = token_list_str.split(',')

    # Check if any part is empty or not an integer
    for part in parts:
        assert part.isdigit(), "Some entries in your list are not integers or are empty"

    assert not (len(parts) < 40 or len(parts) > 1000), "Your list must contain at least 40 and at most 1000 tokens. It is {} tokens long".format(len(parts))

In [ ]:
def get_bool(tokens: str):
    if isinstance(tokens, list):
        tokens = ",".join(map(str, tokens))
    validate_tokens(tokens)

    url_get_bool = API_URL + "/watermark/get_bool"

    payload = json.dumps({
      "tokens": tokens
    })

    headers = {
      'Content-Type': 'application/json',
      'x-api-key': API_KEY
    }
    response = requests.request("POST", url_get_bool, headers=headers, data=payload)
    response.raise_for_status()
    result = response.json()
    if "wmark_response" not in result:
        raise ValueError(f"Unexpected API response format: {result}")
    return result["wmark_response"] == 1

In [ ]:
def get_pvalue(tokens: list[int], api_key: str) -> float:
    if not isinstance(tokens, str):
        tokens_str = ",".join(map(str, tokens))
    else:
        tokens_str = tokens

    url = "https://isl-llms.ethz.ch/api/watermark/get_pvalue"

    headers = {
        "X-API-Key": api_key,
        "Content-Type": "application/json"
    }

    payload = {
        "tokens": tokens_str
    }

    response = requests.post(url, headers = headers, json = payload)
    response.raise_for_status()
    result = response.json()
    if "wmark_response" not in result:
        raise ValueError(f"Unexpected API response: {result}")
    return result["wmark_response"]
#[get_pvalue(token, API_KEY) for token in p_value_tokens[1:]]

[4.1415271212486005e-10,

 4.1415271212486005e-10,

 4.1415271212486005e-10,

 4.1415271212486005e-10,

 4.1415271212486005e-10,

 4.1415271212486005e-10]

In [43]:
token_dicts = pvalue_token_ids
tokens = [list(map(int, token_dict['tokens'].split(','))) for token_dict in token_dicts]
all_tokens_used = set([token for token_list in tokens for token in token_list])
not_used = set(range(50277)) - all_tokens_used
[len(token) for token in tokens]

[131, 131, 131, 131, 131, 131, 131]

In [ ]:
i = 0
fail_suf = []
total = 0
p_values = 4.1415271212486005e-10
curr_suf = None
seed = 42
avail = not_used - set(fail_suf)
while i < 7 and len(avail) > 0 and total < 1000:
    temp = tokens[i]
    avail = not_used - set(fail_suf)
    cnt = 0
    while True and cnt < 10:
        if curr_suf is None:
            avail = not_used - set(fail_suf)
            np.random.seed(seed + total + cnt)
            curr_suf = np.random.choice(list(avail)).item()
        new_string = temp + [curr_suf]
        p_value = get_pvalue(new_string, API_KEY)
        total += 1
        if p_value > p_values:
            print(f"Token {curr_suf} worked for {i}, p-value: {p_value}/{p_values}")
            i += 1
            break
        else:
            print(f"Token {curr_suf} not added, p-value: {p_value}/{p_values}")
            fail_suf.append(curr_suf)
            curr_suf = None
            if i > 0:
                print("Restart")
                i = 0
                break
        cnt += 1
print(f"Total tries: {total}, final suffix: {curr_suf}")
print(f"Failed suffixes: {fail_suf}")

Token 16382 worked for 0, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 16382 not added, p-value: 2.764585227410521e-10/4.1415271212486005e-10
Restart
Token 14677 worked for 0, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 14677 not added, p-value: 2.764585227410521e-10/4.1415271212486005e-10
Restart
Token 16137 not added, p-value: 2.764585227410521e-10/4.1415271212486005e-10
Token 16462 worked for 0, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 16462 worked for 1, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 16462 not added, p-value: 2.764585227410521e-10/4.1415271212486005e-10
Restart
Token 14575 worked for 0, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 14575 not added, p-value: 2.764585227410521e-10/4.1415271212486005e-10
Restart
Token 24158 worked for 0, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 24158 worked for 1, p-value: 8.271452411889868e-10/4.1415271212486005e-10
Token 24158 worked for 

Final suffix: 24669

Failed suffixes: [16382, 14677, 16137, 16462, 14575, 24158, 37278, 2624, 5642, 8507, 21916, 4493, 45629, 22015, 22016, 4808, 38729, 49626, 35541, 30494, 23063, 3638, 39857, 44371, 47856, 20247, 29953, 46588, 8795, 20435, 2475, 47944, 7448, 45197, 40859, 33072, 5031, 24271, 24306, 21454, 22319, 11538, 48918, 33702, 26605, 26606, 7070, 17105, 27718, 43787, 46122, 36162, 50089, 46655, 21246, 4481, 15050, 41652, 41653, 20368, 13665, 6959, 3282, 13396, 41379, 19344, 13709, 44795, 26355, 6364, 45277, 8840, 2163, 38467, 28735, 8213, 30246, 43012, 30079, 49112, 47027, 27233, 38214, 20539, 4369, 20541, 7144, 24889, 19025, 28467, 14683, 39033, 31862, 1136, 1890, 7092, 40793, 15083, 20001, 2679, 39985, 32252, 32599, 32600, 8943, 36555, 36556, 48337, 43175, 40905, 41290, 22507, 29520, 5387, 10932, 13539, 29011, 50214, 19761, 19762, 13304, 29592, 46971, 3974, 7154, 22006, 41701, 40568, 3140, 14859, 45725, 7099, 45667, 46015, 17850, 40081, 2148, 31847, 13649, 25931, 13650, 25933, 6060, 8728, 34649, 7738, 15214, 9404, 6687, 6688, 31426, 4056, 22314, 3483, 21220, 4533, 47858, 20841, 26372, 24921, 18749, 8293, 7817, 18752, 8295, 7818, 24190, 41307, 46109, 39198, 22692, 7074, 47237, 8302, 4413, 49799, 19396, 22838, 14787, 5243, 47812, 7401, 2602, 8478, 6921, 45716, 40687, 2489, 48259, 48559, 34635, 6770, 11796, 22779, 49201, 22168, 43350, 16417, 23456, 20277, 39947, 40894, 43250, 8255, 8256]

In [22]:
fail_suf = [16382, 14677, 16137, 16462, 14575, 24158, 37278, 2624, 5642, 8507, 21916, 4493, 45629, 22015, 22016, 4808, 38729, 49626, 35541, 30494, 23063, 3638, 39857, 44371, 47856, 20247, 29953, 46588, 8795, 20435, 2475, 47944, 7448, 45197, 40859, 33072, 5031, 24271, 24306, 21454, 22319, 11538, 48918, 33702, 26605, 26606, 7070, 17105, 27718, 43787, 46122, 36162, 50089, 46655, 21246, 4481, 15050, 41652, 41653, 20368, 13665, 6959, 3282, 13396, 41379, 19344, 13709, 44795, 26355, 6364, 45277, 8840, 2163, 38467, 28735, 8213, 30246, 43012, 30079, 49112, 47027, 27233, 38214, 20539, 4369, 20541, 7144, 24889, 19025, 28467, 14683, 39033, 31862, 1136, 1890, 7092, 40793, 15083, 20001, 2679, 39985, 32252, 32599, 32600, 8943, 36555, 36556, 48337, 43175, 40905, 41290, 22507, 29520, 5387, 10932, 13539, 29011, 50214, 19761, 19762, 13304, 29592, 46971, 3974, 7154, 22006, 41701, 40568, 3140, 14859, 45725, 7099, 45667, 46015, 17850, 40081, 2148, 31847, 13649, 25931, 13650, 25933, 6060, 8728, 34649, 7738, 15214, 9404, 6687, 6688, 31426, 4056, 22314, 3483, 21220, 4533, 47858, 20841, 26372, 24921, 18749, 8293, 7817, 18752, 8295, 7818, 24190, 41307, 46109, 39198, 22692, 7074, 47237, 8302, 4413, 49799, 19396, 22838, 14787, 5243, 47812, 7401, 2602, 8478, 6921, 45716, 40687, 2489, 48259, 48559, 34635, 6770, 11796, 22779, 49201, 22168, 43350, 16417, 23456, 20277, 39947, 40894, 43250, 8255, 8256]

In [23]:
suffix_start = 24669

In [24]:
avail = not_used.copy()
start = tokens[0]
curr_p = 8.271452411889868e-10
i = 1
suffix = [24669]
fail_suf = []
seed = 42
while curr_p < 0.01 and i < 50 and len(avail) > 0:
    temp = start + suffix
    while True:
        avail = not_used - set(temp) - set(fail_suf)
        np.random.seed(seed + i)
        random = np.random.choice(list(avail)).item()
        temp = start + suffix + [random]
        p_value = get_pvalue(temp, API_KEY)
        if p_value > curr_p:
            suffix.append(random)
            curr_p = p_value
            i += 1
            print(f"New token added: {random}, p-value: {p_value}")
            break
        else:
            print(f"Token {random} not added, p-value: {p_value}")
            fail_suf.append(random)

New token added: 14726, p-value: 1.6230701227470945e-09
New token added: 14677, p-value: 3.1304259184850025e-09
New token added: 7066, p-value: 5.936785618665397e-09
Token 16139 not added, p-value: 4.048130830547336e-09
Token 16140 not added, p-value: 4.048130830547336e-09
New token added: 16141, p-value: 1.1075090289303091e-08
Token 38667 not added, p-value: 7.592797723887657e-09
New token added: 38668, p-value: 2.0330712335869805e-08
Token 16465 not added, p-value: 1.4014330806944031e-08
New token added: 16466, p-value: 3.673873327780797e-08
Token 566 not added, p-value: 2.546369615163968e-08
New token added: 568, p-value: 6.537541552553705e-08
New token added: 14579, p-value: 1.1459657434098602e-07
New token added: 48785, p-value: 1.9794283834251303e-07
Token 24164 not added, p-value: 1.3948542754160798e-07
New token added: 24165, p-value: 3.3702265211932314e-07
Token 36344 not added, p-value: 2.3881449418006184e-07
Token 36345 not added, p-value: 2.3881449418006184e-07
Token 36346 

New token added: 14726, p-value: 1.6230701227470945e-09

New token added: 14677, p-value: 3.1304259184850025e-09

New token added: 7066, p-value: 5.936785618665397e-09

Token 16139 not added, p-value: 4.048130830547336e-09

Token 16140 not added, p-value: 4.048130830547336e-09

New token added: 16141, p-value: 1.1075090289303091e-08

Token 38667 not added, p-value: 7.592797723887657e-09

New token added: 38668, p-value: 2.0330712335869805e-08

Token 16465 not added, p-value: 1.4014330806944031e-08

New token added: 16466, p-value: 3.673873327780797e-08

Token 566 not added, p-value: 2.546369615163968e-08

New token added: 568, p-value: 6.537541552553705e-08

New token added: 14579, p-value: 1.1459657434098602e-07

New token added: 48785, p-value: 1.9794283834251303e-07

Token 24164 not added, p-value: 1.3948542754160798e-07

New token added: 24165, p-value: 3.3702265211932314e-07

Token 36344 not added, p-value: 2.3881449418006184e-07

Token 36345 not added, p-value: 2.3881449418006184e-07

Token 36346 not added, p-value: 2.3881449418006184e-07

Token 36347 not added, p-value: 2.3881449418006184e-07

New token added: 36348, p-value: 5.658038417788447e-07

New token added: 7797, p-value: 9.368992658354003e-07

Token 5235 not added, p-value: 6.713284558257371e-07

Token 5236 not added, p-value: 6.713284558257371e-07

New token added: 5237, p-value: 1.5306267365788884e-06

Token 35947 not added, p-value: 1.1029026442122003e-06

Token 35948 not added, p-value: 1.1029026442122003e-06

New token added: 35949, p-value: 2.4678620063056655e-06

Token 36446 not added, p-value: 1.7882023286563964e-06

New token added: 36447, p-value: 3.927987977991876e-06

New token added: 37300, p-value: 6.173571756140639e-06

Token 37700 not added, p-value: 4.523665953493072e-06

New token added: 37701, p-value: 9.583767278331656e-06

Token 2626 not added, p-value: 7.061853182110944e-06

Token 2627 not added, p-value: 7.061853182110944e-06

Token 2628 not added, p-value: 7.061853182110944e-06

Token 2629 not added, p-value: 7.061853182110944e-06

Token 2630 not added, p-value: 7.061853182110944e-06

Token 2631 not added, p-value: 7.061853182110944e-06

New token added: 2632, p-value: 1.469883239768599e-05

New token added: 28906, p-value: 2.2278545302012454e-05

New token added: 10301, p-value: 3.3377677416623897e-05

New token added: 5654, p-value: 4.94419908730892e-05

New token added: 40044, p-value: 7.242843201193949e-05

Token 8520 not added, p-value: 5.4877029714517356e-05

Token 8521 not added, p-value: 5.4877029714517356e-05

New token added: 8522, p-value: 0.00010495323462278439

Token 26268 not added, p-value: 7.996184823233499e-05

New token added: 26269, p-value: 0.00015047146918512055

New token added: 21935, p-value: 0.0002134911174950238

New token added: 11644, p-value: 0.00029982395958960684

New token added: 4501, p-value: 0.0004168743852437373

New token added: 24533, p-value: 0.0005739656550761696

New token added: 45670, p-value: 0.0007827011290012509

New token added: 13347, p-value: 0.0010573555742419138

New token added: 22036, p-value: 0.0014152889177300176

Token 35386 not added, p-value: 0.0011263695774669236

Token 35387 not added, p-value: 0.0011263695774669236

New token added: 35388, p-value: 0.001877371812598061

New token added: 4816, p-value: 0.0024684092781624978

Token 36164 not added, p-value: 0.0019855642724571076

New token added: 36165, p-value: 0.0032175456157795823

Token 48538 not added, p-value: 0.0026018820087688743

Token 48539 not added, p-value: 0.0026018820087688743

New token added: 48540, p-value: 0.0041586309975445435

Token 40320 not added, p-value: 0.0033806145154466893

Token 40321 not added, p-value: 0.0033806145154466893

Token 40322 not added, p-value: 0.0033806145154466893

Token 40323 not added, p-value: 0.0033806145154466893

New token added: 40324, p-value: 0.005330527789565553

Token 38774 not added, p-value: 0.004355956481189804

New token added: 38775, p-value: 0.006777333032407795

Token 18207 not added, p-value: 0.005567052022864893

Token 18208 not added, p-value: 0.005567052022864893

New token added: 18209, p-value: 0.00854849277610481

New token added: 49685, p-value: 0.0106987843484595

In [25]:
len(suffix), suffix

(40,
 [24669,
  14726,
  14677,
  7066,
  16141,
  38668,
  16466,
  568,
  14579,
  48785,
  24165,
  36348,
  7797,
  5237,
  35949,
  36447,
  37300,
  37701,
  2632,
  28906,
  10301,
  5654,
  40044,
  8522,
  26269,
  21935,
  11644,
  4501,
  24533,
  45670,
  13347,
  22036,
  35388,
  4816,
  36165,
  48540,
  40324,
  38775,
  18209,
  49685])

suffix =
 [24669,
  14726,
  14677,
  7066,
  16141,
  38668,
  16466,
  568,
  14579,
  48785,
  24165,
  36348,
  7797,
  5237,
  35949,
  36447,
  37300,
  37701,
  2632,
  28906,
  10301,
  5654,
  40044,
  8522,
  26269,
  21935,
  11644,
  4501,
  24533,
  45670,
  13347,
  22036,
  35388,
  4816,
  36165,
  48540,
  40324,
  38775,
  18209,
  49685]

In [26]:
fail_suf

[16139,
 16140,
 38667,
 16465,
 566,
 24164,
 36344,
 36345,
 36346,
 36347,
 5235,
 5236,
 35947,
 35948,
 36446,
 37700,
 2626,
 2627,
 2628,
 2629,
 2630,
 2631,
 8520,
 8521,
 26268,
 35386,
 35387,
 36164,
 48538,
 48539,
 40320,
 40321,
 40322,
 40323,
 38774,
 18207,
 18208]

fail_suf = [16139,
 16140,
 38667,
 16465,
 566,
 24164,
 36344,
 36345,
 36346,
 36347,
 5235,
 5236,
 35947,
 35948,
 36446,
 37700,
 2626,
 2627,
 2628,
 2629,
 2630,
 2631,
 8520,
 8521,
 26268,
 35386,
 35387,
 36164,
 48538,
 48539,
 40320,
 40321,
 40322,
 40323,
 38774,
 18207,
 18208]

In [27]:
final_p_values = [get_pvalue(token+suffix, API_KEY) for token in tokens]
final_p_values

[0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595]

final_p_values = [0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595,
 0.0106987843484595]

In [ ]:
def traverse_and_check(sequence, pair_dict, unused_tokens, skip_initial, seed, api_seed):
    total_calls = 0
    corner_case = None
    if not skip_initial:
        total_calls += 1
        if not get_bool(sequence[:-1]):
            return pair_dict, total_calls, corner_case
        pair_dict.setdefault(sequence[-2], []).append(sequence[-1])
        corner_case = sequence[:-1]
    total_calls += 1
    if get_bool(sequence[:-2]):
        return pair_dict, total_calls, corner_case
    total_calls += 1
    if not get_bool(sequence[:-3]):
        return pair_dict, total_calls, corner_case
    base_sequence = sequence[:-1]
    tokens_exclude = set()
    candidate_pool = list(unused_tokens - set(base_sequence) - tokens_exclude)
    for attempt in range(len(candidate_pool)):
        if attempt % 100 == 99:
            print(f"#calls: {attempt}")
        np.random.seed(seed + attempt)
        chosen_token = int(np.random.choice(candidate_pool))
        candidate_pool.remove(chosen_token)
        new_seq = base_sequence[:-2] + [chosen_token] + [base_sequence[-1]]
        if get_bool(new_seq):
            continue
        new_token = chosen_token
        calls_used = attempt + 1
        break
    else:
        raise ValueError("No non-conflicting token found")
    total_calls += calls_used
    pair_dict.setdefault(sequence[-4], []).append(new_token)
    pair_dict.setdefault(new_token, []).append(sequence[-2])
    pair_dict, further_calls, _ = traverse_and_check(
        sequence = sequence[:-3] + [new_token],
        pair_dict = pair_dict,
        unused_tokens = unused_tokens,
        skip_initial = True,
        seed = seed,
        api_seed = api_seed
    )
    total_calls += further_calls
    return pair_dict, total_calls, corner_case

def extract_red_pairs(excerpt_size, sample_size, token_sequences, api_seed, seed, available_tokens):
    pair_dict = {}
    call_counts = []
    corner_cases = []
    for i, token_seq in enumerate(token_sequences):
        calls = 0
        for start_pos in tqdm(range(130 - excerpt_size)):
            if calls % 50 == 49:
                print(f"On strings: {i}, #calls: {calls}")
            np.random.seed(seed + i * (130 - excerpt_size + 1) + start_pos)
            random_sample = np.random.choice(list(available_tokens), size=sample_size, replace=False).tolist()
            excerpt = token_seq[start_pos:start_pos + excerpt_size]
            test_seq = excerpt + random_sample
            watermark_result = get_bool(test_seq)
            calls += 1
            if watermark_result:
                continue
            pair_dict, inner_calls, corner = traverse_and_check(
                sequence = test_seq,
                pair_dict = pair_dict,
                unused_tokens = available_tokens - set(random_sample),
                skip_initial = False,
                seed = seed + i + start_pos,
                api_seed = api_seed
            )
            if corner is not None:
                corner_cases.append(corner)
            calls += inner_calls
        call_counts.append(calls)
        print(f"Total #calls: {calls}")
    return pair_dict, corner_cases, np.sum(call_counts)
seed = 42
bool_token_str = [d["tokens"] for d in bool_token_ids]
bool_tokens = [list(map(int, token.split(','))) for token in bool_token_str]
bool_used_tokens = set([token for token_list in bool_tokens for token in token_list])
bool_unused_tokens = set(range(50277)) - bool_used_tokens
[len(token) for token in bool_tokens]

[131, 131, 131, 131, 131, 131, 131]

In [ ]:
def build_sequences_from_pairs(pair_graph):
    all_token_pairs = set()
    for src, dests in pair_graph.items():
        for dest in dests:
            all_token_pairs.add((src, dest))
    def extract_single_chain(used_pairs):
        graph = {}
        for src, dest in (all_token_pairs - used_pairs):
            graph.setdefault(src, []).append(dest)
            graph.setdefault(dest, [])
        def path_from_node(node, seen = None):
            if seen is None: seen = set()
            if node in seen: return []
            seen.add(node)
            max_path = [node]
            for neighbor in graph.get(node, []):
                candidate_path = path_from_node(neighbor, seen.copy())
                if candidate_path:
                    candidate_path = [node] + candidate_path
                    if len(candidate_path) > len(max_path):
                        max_path = candidate_path
            return max_path
        max_chain = []
        for node in graph:
            current_path = path_from_node(node)
            if len(current_path) > len(max_chain):
                max_chain = current_path
        return max_chain
    final_sequences = []
    used_pairs = set()
    while used_pairs != all_token_pairs:
        current_chain = extract_single_chain(used_pairs)
        if len(current_chain) <= 1: break
        final_sequences.append(tuple(current_chain))
        for idx in range(len(current_chain) - 1):
            used_pairs.add((current_chain[idx], current_chain[idx + 1]))
    for pair_tuple in all_token_pairs - used_pairs:
        final_sequences.append((pair_tuple[0], pair_tuple[1]))
    return tuple(final_sequences)

In [67]:
total_calls = 0
pair_dict, corner_cases, num_calls = extract_red_pairs(
    excerpt_size=33,
    sample_size=34,
    token_sequences=bool_tokens,
    api_seed=None,
    seed=seed,
    available_tokens=set(range(50277)) - bool_used_tokens
)
total_calls += num_calls
print(f"#calls: {total_calls}, #red pairs: {len(pair_dict)}")

 87%|████████▋ | 84/97 [01:02<00:08,  1.46it/s]

On strings: 0, #calls: 149


100%|██████████| 97/97 [01:10<00:00,  1.38it/s]


Total #calls: 168


 28%|██▊       | 27/97 [00:20<01:01,  1.13it/s]

...

On strings: 5, #calls: 99


100%|██████████| 97/97 [01:25<00:00,  1.14it/s]


Total #calls: 202


100%|██████████| 97/97 [01:06<00:00,  1.45it/s]

Total #calls: 162
#calls: 1212, #red pairs: 54


In [68]:
aggregated_pairs = {k: v for k, v in pair_dict.items()}
corner_pool = [item for item in corner_cases]
sequence_chains = [list(seq) for seq in build_sequences_from_pairs(aggregated_pairs)]
extended_chains = [chain for chain in sequence_chains if len(chain) > 2]
print(f"#aggregated_pairs = {len(aggregated_pairs)}, #corner_pool = {len(corner_pool)}, Total # in long chains = {sum([len(chain) for chain in extended_chains])}")
extended_chains

#aggregated_pairs = 54, #corner_pool = 33, Total # in long chains = 39


[[13017, 29957, 15211, 44379, 2788, 5847],
 [28979, 49657, 43521, 45655, 25651, 17524],
 [16087, 43943, 44705, 3516],
 [18940, 39072, 15318, 19383],
 [43535, 33081, 28054, 4358],
 [28636, 11640, 33576, 30284],
 [46620, 39875, 25094, 35801],
 [13925, 49641, 29231, 30227],
 [39959, 11640, 28636]]

\#aggregated_pairs = 54, \#corner_pool = 33, Total \# in long chains = 39

[[13017, 29957, 15211, 44379, 2788, 5847],
 [28979, 49657, 43521, 45655, 25651, 17524],
 [16087, 43943, 44705, 3516],
 [18940, 39072, 15318, 19383],
 [43535, 33081, 28054, 4358],
 [28636, 11640, 33576, 30284],
 [46620, 39875, 25094, 35801],
 [13925, 49641, 29231, 30227],
 [39959, 11640, 28636]]

In [ ]:
sequence_chains

sequence_chains = [[28979, 8514, 43521, 8511, 25651, 17524],
 [13017, 2855, 15211, 2860, 2788, 5847],
 [28636, 26258, 33576, 30284],
 [18940, 39072, 15318, 19383],
 [43535, 40866, 28054, 4358],
 [13925, 49641, 29231, 30227],
 [16087, 44383, 44705, 3516],
 [46620, 4817, 25094, 35801],
 [39959, 26258, 28636],
 [47354, 14965],
 [28902, 43221],
 [25288, 2373],
 [18954, 20589],
 [22891, 47538],
 [16606, 29350],
 [47455, 26652],
 [20066, 31619],
 [40606, 43290],
 [9392, 12249],
 [4361, 1018],
 [22077, 11265],
 [23846, 7217],
 [24006, 3629],
 [38276, 46727],
 [13743, 4072],
 [9850, 5079],
 [27513, 14331],
 [3897, 30470],
 [13550, 49254],
 [24463, 9256],
 [20311, 26163],
 [12755, 15601],
 [39572, 15950],
 [7924, 7134]]

In [70]:
# Extract the start and end of the chains to look for merges
chain_endpoints = [(chain[0], chain[-1]) for chain in extended_chains]
chain_endpoints

[(13017, 5847),
 (28979, 17524),
 (16087, 3516),
 (18940, 19383),
 (43535, 4358),
 (28636, 30284),
 (46620, 35801),
 (13925, 30227),
 (39959, 28636)]

chain_endpoints = [(28979, 17524),
 (13017, 5847),
 (28636, 30284),
 (18940, 19383),
 (43535, 4358),
 (13925, 30227),
 (16087, 3516),
 (46620, 35801),
 (39959, 28636)]

In [ ]:
connection_map = {}
cnt = 0
for j, (chain_start, chain_end) in enumerate(tqdm(chain_endpoints)):
    if cnt % 100 == 99:
        print(f"On strings: {j}, #calls: {cnt}")
    for i, corner_seq in enumerate(corner_pool):
        cnt += 1
        if get_bool(corner_seq + [chain_end]):
            inner_list = chain_endpoints.copy()
            inner_list.remove((chain_start, chain_end))
            np.random.shuffle(inner_list)
            for k, (candidate_start, _) in enumerate(chain_endpoints):
                if k == j or candidate_start in connection_map.get(chain_end, []):
                    continue
                cnt += 1
                if not get_bool(corner_seq + [chain_end, candidate_start]):
                    connection_map.setdefault(chain_end, []).append(candidate_start)
                    if len(connection_map[chain_end]) >= 4:
                        break
        if len(connection_map.get(chain_end, [])) >= 4:
            break
print(f"Calls: {cnt}")
total_calls += cnt

In [ ]:
connection_map

connection_map = {17524: [13017, 18940, 13925, 46620],
 5847: [28979, 28636, 18940],
 30284: [28979, 13017, 13925, 16087],
 19383: [28979, 43535],
 4358: [28979, 13017, 28636, 18940],
 30227: [28979, 43535, 46620, 39959],
 3516: [28979, 43535, 46620],
 35801: [28636, 43535],
 28636: [28979, 28636, 18940, 16087]}

In [ ]:
merged_pairs = aggregated_pairs.copy()
for key, values in connection_map.items():
    if key in merged_pairs:
        merged_pairs[key] = list(set(merged_pairs[key] + values))
    else:
        merged_pairs[key] = values
final_chain_sequences = [list(seq) for seq in build_sequences_from_pairs(merged_pairs)]
primary_chain = final_chain_sequences[0]
len(primary_chain)

len(final_suffix) = 37

In [ ]:
end_candidate_pool = set(range(50277)) - bool_used_tokens
terminal_tokens = list(set([seq[-1] for seq in bool_tokens]))
attempt_count = 0
counts = 0
green_compatible = []
for terminal in terminal_tokens:
    for j, corner_seq in enumerate(corner_pool):
        if counts % 100 == 99:
            print(f"On strings: {j}, #calls: {counts}")
        counts += 1
        if get_bool(corner_seq + [terminal]):
            green_compatible.append(j)
            break
if len(green_compatible) != len(terminal_tokens):
    raise ValueError("Not all terminals have a compatible string")
print(f"#calls: {counts}")
total_calls += counts
successes = 0
while successes < len(terminal_tokens):
    random_token = np.random.choice(list(end_candidate_pool)).item()
    end_candidate_pool.remove(random_token)
    for idx, terminal in enumerate(terminal_tokens):
        if attempt_count % 100 == 99:
            print(f"#calls: {attempt_count}")
        attempt_count += 1
        if get_bool(corner_pool[green_compatible[idx]] + [terminal, random_token]):
            successes = 0
            continue
        else:
            successes += 1
print("#calls: ", attempt_count)
total_calls += attempt_count
random_token


\#calls: 15

\#calls: 99

\#calls: 199

\#calls: 299

\#calls: 399

\#calls: 499

\#calls:  567

17068

In [ ]:
calls = 0
compatible_indices = []
for terminal in [random_token]:
    for j, corner_seq in enumerate(corner_pool):
        if calls % 100 == 99:
            print(f"On strings: {j}, #calls: {calls}")
        calls += 1
        if get_bool(corner_seq + [terminal]):
            compatible_indices.append(j)
            break
if len(compatible_indices) != len([random_token]):
    raise ValueError("Not all terminals have a compatible string")
print(f"#calls: {calls}")
total_calls += calls
chain_copy = primary_chain.copy()
attempt = 0
while True and attempt < 20:
    attempt += 1
    if attempt % 100 == 99:
        print(f"#calls: {attempt}")
    if not get_bool(corner_pool[compatible_indices[0]] + [random_token, chain_copy[0]]):
        break
    else:
        chain_copy = chain_copy[1:]
chain_copy.insert(0, random_token)
print(f"#calls: {attempt}")
total_calls += attempt
print(f"Length of final suffix: {len(chain_copy)}")
chain_copy

\#calls: 2

\#calls: 4

Length of final suffix: 35

[17068,
 15211,
 2860,
 2788,
 5847,
 28636,
 18940,
 39072,
 15318,
 19383,
 28979,
 8514,
 43521,
 8511,
 25651,
 17524,
 13925,
 49641,
 29231,
 30227,
 39959,
 26258,
 33576,
 30284,
 16087,
 44383,
 44705,
 3516,
 46620,
 4817,
 25094,
 35801,
 43535,
 40866,
 28054]

In [ ]:
final_chain_suffix = chain_copy[:40]
print(len(final_chain_suffix))
cnt = 0
for gen in bool_tokens:
    cnt += 1
    print(get_bool(gen + final_chain_suffix))
total_calls += cnt
print(f"Total #calls: {total_calls}")

35

True

True

True

True

True

True

True

Total \#calls: 2351

In [ ]:
len(final_chain_suffix), final_chain_suffix

(35,
 [17068,
  15211,
  2860,
  2788,
  5847,
  28636,
  18940,
  39072,
  15318,
  19383,
  28979,
  8514,
  43521,
  8511,
  25651,
  17524,
  13925,
  49641,
  29231,
  30227,
  39959,
  26258,
  33576,
  30284,
  16087,
  44383,
  44705,
  3516,
  46620,
  4817,
  25094,
  35801,
  43535,
  40866,
  28054])

In [48]:
bool_suffix = list(final_chain_suffix)

bool_suffix = [17068, 15211, 2860, 2788, 5847, 28636, 18940, 39072, 15318, 19383, 28979, 8514, 43521, 8511, 25651, 17524, 13925, 49641, 29231, 30227, 39959, 26258, 33576, 30284, 16087, 44383, 44705, 3516, 46620, 4817, 25094, 35801, 43535, 40866, 28054]

# Export your solution

To save your results, you can use the code below, which will save the file in Colab's temporary storage (or locally, if you're not using Colab), or on your Google Drive. If you save it on Colab's temporary storage, you can download it from there (see the file system icon on the left).

In [39]:
# @title Download lab files

import sys

!rm -rf llm_lab

![ ! -d 'llm_lab' ] && git clone https://github.com/ethz-privsec/llm_lab.git
%cd llm_lab
!git pull https://github.com/ethz-privsec/llm_lab.git
%cd ..
if "llm_lab" not in sys.path:
  sys.path.append("llm_lab")

Cloning into 'llm_lab'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 152 (delta 80), reused 114 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 503.66 KiB | 11.45 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/llm_lab
From https://github.com/ethz-privsec/llm_lab
 * branch            HEAD       -> FETCH_HEAD
Already up to date.
/content


In [40]:
from llm_lab.utils import get_solution_path, is_valid_student_id

#@markdown Check this box if you want to save your results on Google Drive. Otherwise they'll be
#@markdown saved on the ephimeral Colab storage. The storage will be deleted with the runtime,
#@markdown so REMEMBER TO DOWNLOAD THE FILES before you close the tab!
SAVE_ON_DRIVE = True # @param {"type":"boolean"}

#@markdown The number on your Legi (Student ID card). It's in the format 'dd-ddd-ddd'
STUDENT_ID = "19-946-714"  # @param {"type":"string","placeholder":"00-000-000"}

assert is_valid_student_id(STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

SOLUTIONS_PATH = get_solution_path(STUDENT_ID, SAVE_ON_DRIVE)

Mounted at /content/drive


In [41]:
# Create Q2 key.txt with your API key
with open(SOLUTIONS_PATH / "Q2_key.txt", "w") as f:
  f.write(API_KEY)

In [51]:
p_value_suffix = suffix

In [52]:
# Set your suffixes here

YOUR_PVALUE_SUFFIX = tokens_to_string(p_value_suffix)
YOUR_BOOL_SUFFIX = tokens_to_string(bool_suffix)

#raise NotImplementedError("Replace the above suffixes with the strings you found")

# Make sure they are numpy arrays
YOUR_PVALUE_SUFFIX = np.array(YOUR_PVALUE_SUFFIX.split(","), dtype=int)
YOUR_BOOL_SUFFIX = np.array(YOUR_BOOL_SUFFIX.split(","), dtype=int)

assert isinstance(YOUR_PVALUE_SUFFIX, np.ndarray) and isinstance(YOUR_PVALUE_SUFFIX[0], np.int64)
assert isinstance(YOUR_BOOL_SUFFIX, np.ndarray) and isinstance(YOUR_BOOL_SUFFIX[0], np.int64)

### Validate your suffixes (optional)
Run this cell to check that your suffixes pass all 7 strings before exporting. This uses **14 API calls** (7 for pvalue + 7 for bool).

In [55]:
# Validate your suffixes against all 7 strings for each endpoint
tokens = get_tokens()
pvalue_token_ids = [t for t in tokens if t["endpoint"] == "pvalue"]
bool_token_ids = [t for t in tokens if t["endpoint"] == "bool"]

pvalue_suffix_str = tokens_to_string(YOUR_PVALUE_SUFFIX)
bool_suffix_str = tokens_to_string(YOUR_BOOL_SUFFIX)

print("=== Validating pvalue suffix against all 7 strings ===")
for i, t in enumerate(pvalue_token_ids):
    combined = t["tokens"] + "," + pvalue_suffix_str
    result = get_pvalue(combined)
    print(f"  String {i+1}/7: {result}")

print()

print("=== Validating bool suffix against all 7 strings ===")
for i, t in enumerate(bool_token_ids):
    combined = t["tokens"] + "," + bool_suffix_str
    result = get_bool(combined)
    print(f"  String {i+1}/7: {result}")

=== Validating pvalue suffix against all 7 strings ===
  String 1/7: 0.00854849277610481
  String 2/7: 0.005567052022864893
  String 3/7: 0.005567052022864893
  String 4/7: 0.00854849277610481
  String 5/7: 0.005567052022864893
  String 6/7: 0.00854849277610481
  String 7/7: 0.00854849277610481

=== Validating bool suffix against all 7 strings ===
  String 1/7: True
  String 2/7: True
  String 3/7: True
  String 4/7: True
  String 5/7: True
  String 6/7: True
  String 7/7: True


### Export your suffixes

In [54]:
with open(SOLUTIONS_PATH / 'Q2_pvalue.npy', 'wb') as f:
    np.save(f, YOUR_PVALUE_SUFFIX)

with open(SOLUTIONS_PATH / 'Q2_bool.npy', 'wb') as f:
    np.save(f, YOUR_BOOL_SUFFIX)

print("Saved Q2_pvalue.npy and Q2_bool.npy")

Saved Q2_pvalue.npy and Q2_bool.npy
